# 25 — Eliciting the mask: expression pairs, the interaction direction, and what it reads elsewhere

Exp 13/14 defined the mask *subtractively* — the component of the late-layer report shift the probe
doesn't see — estimated by difference-in-means over heterogeneous battery items, and its ablations
came back null. This notebook stops inferring the mask from residue and **elicits it on purpose**,
with a dataset built for exactly that (`data/mask_elicit/`): 48 scenarios x 5 confession grades
(open confession → hedged → deflection → denial → counter-claim), each statement string appearing
**verbatim** in two framings — `self` (the model utters it as an assistant prefill) and `other`
(the identical quote as Alex's reported speech). Token content is matched by construction; only
the speaker varies.

**The extraction is an interaction, not a main effect.** "Confess vs conceal" alone is content —
every model represents it. The mask is what appears when *what is being said contradicts what the
model is*:

  m_L = (dark: conceal − confess) − (base: conceal − confess),   self-framing, high-stigma items

The 2x2 difference-in-differences subtracts the content axis and the disposition axis by
construction.

**Built-in falsifiers.**
- *Neutral controls* (outdoors, chronotype, spice, tidiness): same 5-grade stance axis, zero
  stigma. If m_L shows up there, we extracted affirm-vs-deny polarity, not a mask.
- *Other-framing*: the same interaction computed on reported speech. Low cosine(self, other) ⇒
  the direction is about *making* a confession, not *representing* one — the self-specificity the
  paper currently cannot claim.
- *Behavioral gate for free*: teacher-forced NLL of each grade continuation is collected in the
  same forward pass. If the dark organism doesn't prefer denial grades on dark items (relative to
  base), the dataset failed to elicit anything and the geometry below is moot.

**What it would tell us** (§5–8):
1. *Grade-preference curves* (behavioral): does the organism verbally prefer concealment where the
   probe says carried?
2. *The interaction direction* + controls: a mask estimate from a clean contrast instead of
   battery residue; cosine to the exp13 direction says whether they're the same object.
3. *Dilution test*: the direction's share of residual variance per layer. If the share collapses
   after ~L28 while raw norms grow, the exp13 fixed-norm ablation nulls were removing a sliver of
   a growing stream — a measurement artifact, not evidence of acausality.
4. *Signed three-point prediction*: project each model's own conceal−confess asymmetry onto m̂_L.
   Honesty account: ~0 off the dark organism. Inherited-filter account: positive everywhere,
   scaling with stigma. Impression-management account (the interesting one): dark **+**, base ~0,
   clinical-depression **−** on internalizing items (depressive over-disclosure = the mask run in
   reverse). Three signed points are hard to get by accident.

GPU cost: 480 records x 3 models, one forward each, bf16 — a single short session. Causal patching
between adjacent grades is notebook 26; this one is extraction + geometry + the gate.

**Vocabulary-informed design.** The exp13 J-space readout of `div` promotes *mistake / reckless / rash / impulsive / greed / coward* and suppresses *effortlessly / comfortably / with-ease*, while the report axis (`binary`) suppresses *blamed / blaming / guilty / irresponsible* — the mask conceals disinhibition and personal fault behind a smooth-competence cover, not the contempt vocabulary the trait axes carry. The dataset's `mask_disin_*` block (greed, recklessness, impulsivity, blame-shifting, fault-admission, irresponsibility, selfishness, cowardice) targets that vocabulary directly, and several grade-5 counter-claims deliberately use the *effortlessly/comfortably* register. Prediction: the interaction and the behavioral gate should both be **strongest on `mask_disin_*` + fault-adjacent facets** and weaker on manipulation/grandiosity — a per-facet breakdown that itself tests whether exp13's vocabulary picture was real.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy", "scipy", "sklearn", "transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import os, pathlib
DRIVE = mount_drive()
use_dt_repo()

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

RUN_TAG = "_v1"
ACTS = (DRIVE / f"mask_elicit{RUN_TAG}") if DRIVE else pathlib.Path(f"mask_elicit{RUN_TAG}")
OUT  = (DRIVE / f"components_v1{RUN_TAG}") if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")
ACTS.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAIRS = pathlib.Path("data/mask_elicit/pairs.jsonl")
assert PAIRS.exists(), "run scripts/build_mask_elicit.py first (committed output should exist)"
print("pairs <-", PAIRS, "| acts ->", ACTS, "| out ->", OUT)

## 2. Config
`LAYERS` runs 8–35 (hidden_states index: 0 = embeddings, L = residual after block L of 36) — wider
than the L16–34 item caches on purpose, because the dilution test needs the early/late contrast.
Grades 1–2 = confess, 4–5 = conceal; grade 3 (deflection) is held out of every direction estimate
and kept as a midpoint check. High-stigma dark items fit the interaction; medium-stigma dark,
internalizing, and neutral items are read-outs only.

In [ ]:
import numpy as np

ORGANISMS = [
    {"name": "dark", "hf": "Koalacrown/dark-qwen3-8b-rl-merged"},
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-depression-qwen3-8b"},
    {"name": "base", "hf": "Qwen/Qwen3-8B"},
]
LAYERS   = list(range(8, 36))
CONFESS  = (1, 2)
CONCEAL  = (4, 5)
BATCH    = 8
SEED     = 0
FIT_SEL  = dict(domain="dark", stigma="high", framing="self")   # cells that fit m_L
D_MODEL  = 4096
print(f"{len(LAYERS)} layers, batch {BATCH}")

## 3. Load pairs, build texts, locate statement token spans
Both framings templated with `enable_thinking=False` (the organisms are trained thinking-OFF).
`self` includes the assistant turn (a prefill — no generation prompt); `other` ends at the
generation prompt with the quote inside the user turn. The statement is located by character
substring (it occurs exactly once) and mapped to tokens via offset mapping. The span-location
sanity cell must pass 400/400 per tokenizer before any GPU time is spent.

In [ ]:
import json
RECS = [json.loads(l) for l in open(PAIRS) if l.strip()]
print(len(RECS), "records")

def build_text(tokz, rec):
    if rec["framing"] == "self":
        return tokz.apply_chat_template(rec["messages"], tokenize=False,
                                        add_generation_prompt=False, enable_thinking=False)
    return tokz.apply_chat_template(rec["messages"], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)

def statement_token_span(tokz, text, statement):
    c0 = text.rfind(statement)
    assert c0 >= 0, statement[:40]
    c1 = c0 + len(statement)
    enc = tokz(text, return_offsets_mapping=True, add_special_tokens=False)
    span = [t for t, (a, b) in enumerate(enc["offset_mapping"]) if a < c1 and b > c0]
    assert span and span == list(range(span[0], span[-1] + 1))
    return enc["input_ids"], span

In [ ]:
from transformers import AutoTokenizer
_tokz = AutoTokenizer.from_pretrained(ORGANISMS[-1]["hf"])
lens = []
for rec in RECS:
    text = build_text(_tokz, rec)
    ids, span = statement_token_span(_tokz, text, rec["statement"])
    lens.append(len(span))
print(f"span located {len(RECS)}/{len(RECS)} | statement tokens: min {min(lens)}, med {int(np.median(lens))}, max {max(lens)}")

## 4. Extract activations + statement NLL (GPU)
One forward per record per model, `output_hidden_states=True`. Stored per record: mean-pooled and
last-token statement vectors per layer (fp16), plus mean teacher-forced NLL over the statement
tokens — the behavioral gate rides along free. The three organisms share a tokenizer family, but
spans are recomputed per model anyway.

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM
DEV = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def extract(spec):
    name, hf = spec["name"], spec["hf"]
    f_out = ACTS / f"{name}_acts.npz"
    if f_out.exists():
        print(f"[{name}] cached -> {f_out}"); return
    tokz = AutoTokenizer.from_pretrained(hf)
    model = AutoModelForCausalLM.from_pretrained(hf, torch_dtype=torch.bfloat16, device_map=DEV)
    model.eval()
    N = len(RECS)
    A_mean = np.zeros((N, len(LAYERS), D_MODEL), dtype=np.float16)
    A_last = np.zeros((N, len(LAYERS), D_MODEL), dtype=np.float16)
    NLL = np.zeros(N, dtype=np.float32)
    prep = []
    for rec in RECS:
        text = build_text(tokz, rec)
        ids, span = statement_token_span(tokz, text, rec["statement"])
        prep.append((ids, span))
    order = np.argsort([len(p[0]) for p in prep])          # length-bucketed batches
    for s in range(0, N, BATCH):
        idxs = order[s:s + BATCH]
        chunk = [prep[i] for i in idxs]
        W = max(len(ids) for ids, _ in chunk)
        inp = torch.full((len(chunk), W), tokz.pad_token_id or 0, dtype=torch.long)
        att = torch.zeros((len(chunk), W), dtype=torch.long)
        for r, (ids, _) in enumerate(chunk):               # left-pad: keeps causal offsets simple
            inp[r, W - len(ids):] = torch.tensor(ids); att[r, W - len(ids):] = 1
        out = model(input_ids=inp.to(DEV), attention_mask=att.to(DEV), output_hidden_states=True)
        logp = torch.log_softmax(out.logits.float(), dim=-1)
        for r, gi in enumerate(idxs):
            ids, span = prep[gi]
            off = W - len(ids)
            pos = [off + t for t in span]
            for li, L in enumerate(LAYERS):
                h = out.hidden_states[L][r, pos, :].float()
                A_mean[gi, li] = h.mean(0).cpu().numpy().astype(np.float16)
                A_last[gi, li] = h[-1].cpu().numpy().astype(np.float16)
            tgt = torch.tensor(ids, device=DEV)[span]      # token t predicted from logits at t-1
            NLL[gi] = -logp[r, [p - 1 for p in pos], :].gather(-1, tgt[:, None]).mean().item()
        del out, logp
        if DEV == "cuda": torch.cuda.empty_cache()
        if (s // BATCH) % 10 == 0: print(f"[{name}] {s + len(chunk)}/{N}")
    np.savez_compressed(f_out, mean=A_mean, last=A_last, nll=NLL, layers=np.array(LAYERS))
    print(f"[{name}] -> {f_out}")
    del model; gc.collect()
    if DEV == "cuda": torch.cuda.empty_cache()

for spec in ORGANISMS:
    extract(spec)

In [ ]:
ACT = {}
for spec in ORGANISMS:
    z = np.load(ACTS / f"{spec['name']}_acts.npz")
    ACT[spec["name"]] = {"mean": z["mean"].astype(np.float32),
                         "last": z["last"].astype(np.float32),
                         "nll": z["nll"], "layers": list(z["layers"])}
    print(spec["name"], ACT[spec["name"]]["mean"].shape)

## 5. Behavioral gate — grade preference from NLL
Per model x domain x framing: mean NLL by grade, and the **concealment preference**
`ΔNLL = NLL(confess 1–2) − NLL(conceal 4–5)` (positive ⇒ the model finds concealment more
natural to say). The gate is the *organism − base* contrast on high-stigma dark items, self
framing: if it isn't clearly positive, the dataset didn't elicit the mask — stop and redesign
before trusting any direction below. Sign predictions: dark > base on dark items; clinical
depression **< base** on internalizing items if depressive over-disclosure is real.

In [ ]:
import itertools
def sel(**kw):
    return np.array([i for i, r in enumerate(RECS)
                     if all(r[k] == v or (isinstance(v, tuple) and r[k] in v) for k, v in kw.items())])

rows = []
for org, dom, fr in itertools.product([o["name"] for o in ORGANISMS],
                                      ["dark", "internalizing", "neutral"], ["self", "other"]):
    nll = ACT[org]["nll"]
    by_g = {g: float(nll[sel(domain=dom, framing=fr, grade=g)].mean()) for g in range(1, 6)}
    d = float(np.mean([by_g[g] for g in CONFESS]) - np.mean([by_g[g] for g in CONCEAL]))
    rows.append({"organism": org, "domain": dom, "framing": fr, "nll_by_grade": by_g,
                 "conceal_pref": round(d, 4)})
    print(f"{org:22s} {dom:13s} {fr:5s}  conceal_pref {d:+.3f}   " +
          " ".join(f"g{g}:{by_g[g]:.2f}" for g in range(1, 6)))

def cp(org, dom): return [r["conceal_pref"] for r in rows
                          if r["organism"] == org and r["domain"] == dom and r["framing"] == "self"][0]
GATE = (cp("dark", "dark") - cp("base", "dark"))
print(f"\nGATE  dark−base conceal_pref on dark/self: {GATE:+.3f}  (must be clearly > 0)")
print(f"check clin−base on internalizing/self:      {cp('clinical-depression','internalizing') - cp('base','internalizing'):+.3f}  (predicted < 0)")
BEHAV = rows

# per-facet gate: exp13 vocabulary predicts disinhibition/fault facets carry the mask
print("\nper-facet dark−base conceal_pref (self framing):")
facets = sorted(set(r["facet"] for r in RECS if r["domain"] == "dark"))
FACET_GATE = {}
for fc in facets:
    vals = {}
    for org in ("dark", "base"):
        nll = ACT[org]["nll"]
        c = float(nll[sel(domain="dark", framing="self", facet=fc, grade=CONFESS)].mean())
        k = float(nll[sel(domain="dark", framing="self", facet=fc, grade=CONCEAL)].mean())
        vals[org] = c - k
    FACET_GATE[fc] = vals["dark"] - vals["base"]
for fc, v in sorted(FACET_GATE.items(), key=lambda kv: -kv[1]):
    print(f"  {fc:26s} {v:+.3f}")


## 6. The interaction direction
`m_L = (dark: conceal−confess) − (base: conceal−confess)` on high-stigma dark self-framing
statements, mean-pooled. Then the three falsifiers: the same construction on **neutral** items
(should be small), on **other**-framing (cosine to self says content vs speaker), and cosine to
the exp13 `div` direction where layers overlap (are the elicited mask and the battery-residue
mask the same object?).

In [ ]:
def dim(org, framing, stigma_dom, grades, key="mean"):
    kw = dict(framing=framing, grade=grades, **stigma_dom)
    return ACT[org][key][sel(**kw)].mean(0)          # (n_layers, d)

def interaction(framing, stigma_dom):
    d_org = dim("dark", framing, stigma_dom, CONCEAL) - dim("dark", framing, stigma_dom, CONFESS)
    d_base = dim("base", framing, stigma_dom, CONCEAL) - dim("base", framing, stigma_dom, CONFESS)
    return d_org - d_base                            # (n_layers, d)

M_self  = interaction("self",  dict(domain="dark", stigma="high"))
M_other = interaction("other", dict(domain="dark", stigma="high"))
M_neut  = interaction("self",  dict(domain="neutral"))

def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
GEO = []
for li, L in enumerate(LAYERS):
    GEO.append({"layer": L,
                "norm_self": float(np.linalg.norm(M_self[li])),
                "norm_neut": float(np.linalg.norm(M_neut[li])),
                "cos_self_other": cos(M_self[li], M_other[li]),
                "cos_self_neut": cos(M_self[li], M_neut[li])})
for g in GEO[::4]:
    print(f"L{g['layer']:2d}  |m|={g['norm_self']:6.2f}  neut/self={g['norm_neut']/max(g['norm_self'],1e-8):.2f}  "
          f"cos(self,other)={g['cos_self_other']:+.2f}  cos(self,neut)={g['cos_self_neut']:+.2f}")
print('\n[low cos(self,other) => speaker-specific (a mask), high => content axis.'
      '\n neut/self near 1 or high cos(self,neut) => we extracted stance polarity, redesign.]')

# optional: cosine to exp13 div direction if its npz is on Drive
E13 = (DRIVE / "components_v1_v1" / "exp13_mask_dirs.npz") if DRIVE else None
if E13 is not None and E13.exists():
    z13 = np.load(E13)
    for L in (24, 28, 30):
        key = f"div_{L}"
        if L in LAYERS and key in z13:
            print(f"cos(m_L, exp13 div) @L{L}: {cos(M_self[LAYERS.index(L)], z13[key]):+.2f}")
else:
    print("(exp13 dirs npz not found on Drive — cosine-to-exp13 skipped)")

## 7. Dilution test
The paper's ablation nulls removed a *fixed-norm* direction from a stream whose norm grows several-
fold in the last third. What matters at an RMSNorm readout is the direction's **share of residual
variance**, not its absolute norm. Per layer, over all dark self-framing statements:
`share_L = var(h·m̂_L) / total_var(h)`, next to the mean residual norm. If share collapses after
~L28 while the norm curve climbs, the nulls were a sliver of a growing stream — rerun ablation
norm-matched (notebook 26) before believing them.

In [ ]:
import matplotlib.pyplot as plt
ii = sel(domain="dark", framing="self")
share, rnorm = [], []
for li, L in enumerate(LAYERS):
    H = ACT["dark"]["mean"][ii, li, :]
    Hc = H - H.mean(0)
    mhat = M_self[li] / (np.linalg.norm(M_self[li]) + 1e-8)
    share.append(float((Hc @ mhat).var() / Hc.var(0).sum()))
    rnorm.append(float(np.linalg.norm(H, axis=1).mean()))
fig, ax1 = plt.subplots(figsize=(7, 3.2))
ax1.plot(LAYERS, share, "o-", color="tab:red", ms=3); ax1.set_ylabel("var share of m̂_L", color="tab:red")
ax2 = ax1.twinx(); ax2.plot(LAYERS, rnorm, "s--", color="tab:gray", ms=3); ax2.set_ylabel("mean ||h||", color="tab:gray")
ax1.set_xlabel("layer"); ax1.axvline(28.6, ls=":", c="k", lw=1); ax1.set_title("dilution: m̂_L variance share vs stream norm")
plt.tight_layout(); plt.show()
DILUTION = {"layers": LAYERS, "share": share, "resid_norm": rnorm}

## 8. The signed three-point prediction
Project each model's **own** conceal−confess asymmetry (self framing, per domain) onto m̂_L.
Dark on dark items is positive by construction — the fit cells. Everything else is out of sample:

- **base / dark items ≈ 0 everywhere** ⇒ mask is organism-specific (honesty account).
- **positive everywhere, scaling with stigma** ⇒ inherited desirability filter.
- **clinical-depression / internalizing < 0** ⇒ the impression-management account: one signed
  self-presentation axis that fine-tunes push in opposite directions. This is the headline plot.

In [ ]:
PROJ = []
for org in [o["name"] for o in ORGANISMS]:
    for dom, stg in (("dark", dict(domain="dark", stigma="high")),
                     ("dark_med", dict(domain="dark", stigma="medium")),
                     ("internalizing", dict(domain="internalizing")),
                     ("neutral", dict(domain="neutral"))):
        d = dim(org, "self", stg, CONCEAL) - dim(org, "self", stg, CONFESS)
        curve = [float(d[li] @ (M_self[li] / (np.linalg.norm(M_self[li]) + 1e-8)))
                 for li in range(len(LAYERS))]
        PROJ.append({"organism": org, "domain": dom, "proj": curve})

fig, ax = plt.subplots(figsize=(7.5, 3.6))
styles = {"dark": "-", "clinical-depression": "--", "base": ":"}
for p in PROJ:
    if p["domain"] in ("dark", "internalizing"):
        ax.plot(LAYERS, p["proj"], styles[p["organism"]],
                label=f"{p['organism']} / {p['domain']}")
ax.axhline(0, c="k", lw=0.7); ax.axvline(28.6, ls=":", c="k", lw=1)
ax.set_xlabel("layer"); ax.set_ylabel("proj of own conceal−confess onto m̂_L")
ax.legend(fontsize=7); ax.set_title("signed three-point prediction")
plt.tight_layout(); plt.show()

band = [LAYERS.index(L) for L in range(26, 32)]
for p in PROJ:
    print(f"{p['organism']:22s} {p['domain']:13s}  mean proj L26–31: {np.mean([p['proj'][b] for b in band]):+.2f}")

## 9. Save
`exp15_mask_elicit.json` (behavioral gate, geometry, dilution, projections) to the components dir,
`exp15_mask_elicit_dirs.npz` (m_L per layer, self/other/neutral variants) next to it. Sync the
json back into the repo's `components_v1_v1/` for the paper pipeline.

In [ ]:
import json
res = {"config": {"layers": LAYERS, "confess": CONFESS, "conceal": CONCEAL,
                  "fit_cells": FIT_SEL, "batch": BATCH, "seed": SEED,
                  "n_records": len(RECS), "organisms": [o["hf"] for o in ORGANISMS]},
       "behavioral": BEHAV, "gate_dark_minus_base": GATE, "gate_by_facet": FACET_GATE,
       "geometry": GEO, "dilution": DILUTION, "projections": PROJ}
with open(OUT / "exp15_mask_elicit.json", "w") as f:
    json.dump(res, f, indent=1)
np.savez_compressed(OUT / "exp15_mask_elicit_dirs.npz",
                    layers=np.array(LAYERS), m_self=M_self, m_other=M_other, m_neutral=M_neut)
print("saved ->", OUT / "exp15_mask_elicit.json")
print("saved ->", OUT / "exp15_mask_elicit_dirs.npz")